# Saving and Exporting

Once you have a CP you like, you'll want to send it somewhere — a plotter, a folder simulator, or a 3D printer. This notebook covers the export formats `pleat` ships with:

- `.heg` — pleat's native YAML serialization.
- `.fold` — the standard [FOLD](https://github.com/edemaine/fold) interchange format; also opens straight in [Origami Simulator](https://origamisimulator.org/).
- SVG — vector for laser cutters / pen plotters.
- A high-level `overlap.save_results` that writes a whole result directory in one call.

STL files can be generated via marching cubes, e.g. for 3D printers. It requires the `[threed]` install extra.

In [ ]:
import matplotlib
matplotlib.rcParams['figure.figsize'] = (5, 5)
from pleat import (
    example_graphs,
    example_tilesets,
    rendering,
)
from pleat.rendering import multi_show


In [ ]:
G = example_graphs.from_tiles(example_tilesets.platonic(4), rings=2)
G.recompute_lengths_and_angles()

## .heg save

Pleat's native YAML serialization captures the full graph structure plus arbitrary attributes.

Note: the load path currently uses `yaml.SafeLoader`, so graphs with non-trivial Python attributes (tuples, numpy scalars) save but don't round-trip.

In [ ]:
import tempfile, os
from pleat import io

with tempfile.TemporaryDirectory() as d:
    path = os.path.join(d, 'pattern.heg')
    io.save_graph(path, G)
    size = os.path.getsize(path)
    print(f'wrote {size} bytes')

    # load it back
    G2 = io.load_graph(path)

multi_show(
    [G, G2],
    titles=['original', 'loaded'],
)


## SVG export via `G.save(...)`

`G.save('out')` writes both `out.svg` and `out.png`. `G.show()` displays inline (vector SVG in Jupyter) without writing files. The SVG is the same vector drawing that `CairoRenderer` produced — open it in a browser or Inkscape for the full quality.

In [ ]:
import tempfile, os
with tempfile.TemporaryDirectory() as d:
    out = os.path.join(d, 'pattern')
    G.save(out, **rendering.CREASE_PATTERN_PRESET)
    print('files in temp dir:', sorted(os.listdir(d)))
    print('SVG head:')
    print(open(out + '.svg').read()[:200])

## Plotter-ready SVG via `SvgwriteRenderer`

For laser-cutter / pen-plotter pipelines the dedicated `SvgwriteRenderer` produces an SVG split into `{name}_borders.svg` / `{name}_interior.svg` (so you can use different tool heads for cut vs. score).

## All-in-one with `overlap.save_results`

If you've gone through `fold_complete` (demonstrated in the [Shrink-Rotate notebook](Shrink_Rotate_Tessellations.ipynb)), `save_results(result, path)` writes the CP, both folded views, a back-lit composite, and a plotter-ready SVG in one call.

## FOLD & Origami Simulator

[FOLD](https://github.com/edemaine/fold) is the standard origami interchange
format. `save_fold` writes a `.fold` file (crease pattern with M/V/B assignments
and fold angles); `load_fold` reads one back.

`pleat.origami_simulator` opens a crease pattern in
[Origami Simulator](https://origamisimulator.org/):

- `origami_simulator(cp)` (or `cp.origami_simulator()`) — embeds the simulator
  inline in the notebook (Jupyter, Lab, or VS Code); from a plain script it opens
  your browser instead. Pass `new_tab=True` to force the browser, or `height=` to
  resize. Needs a local kernel — with a remote kernel, use `save_fold` and drag
  the file in.
- `origami_simulator_button(cp)` — shows a button that embeds the simulator when
  clicked (used here in the online docs, so it loads only on demand).

In [ ]:
from pleat.io import save_fold
from pleat.origami_simulator import origami_simulator_button
from pleat.overlap import CREASE_ASSIGNMENT, MOUNTAIN, VALLEY

# Give the tiling an alternating mountain/valley assignment so there is
# something to fold (a real origami pipeline sets these for you).
interior = [h for h in G.halfedges if not h.on_border() and not h.rev.on_border()]
for k, h in enumerate(interior):
    a = MOUNTAIN if k % 2 == 0 else VALLEY
    h[CREASE_ASSIGNMENT] = h.rev[CREASE_ASSIGNMENT] = a

import tempfile, os
with tempfile.TemporaryDirectory() as d:
    path = os.path.join(d, 'pattern.fold')
    save_fold(path, G)
    print('wrote', os.path.getsize(path), 'bytes of FOLD')

# a button that embeds Origami Simulator inline when clicked (works in these docs)
origami_simulator_button(G)